# Data Exploration — JobGraph Dataset

This notebook explores the structured job data in `jobs.parquet` (481 extracted job postings from 8 companies), examining distributions of companies, seniority levels, skills, salaries, and role families.

**Data source:** `data/processed/jobs.parquet` — produced by the extraction pipeline from scraped career pages.

In [ ]:
import ast
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

DATA_DIR = Path("..") / "data" / "processed"
print("Data directory:", DATA_DIR.resolve())

## 1. Load Data

In [ ]:
df = pd.read_parquet(DATA_DIR / "jobs.parquet")
print(f"Shape: {df.shape}")
print()
print("Columns & dtypes:")
print(df.dtypes)
print()
df.head(3)

In [ ]:
# Parse required_skills — parquet stores list columns as numpy arrays, not Python lists.
# Also handle string repr or native list for robustness.
def parse_skills(val):
    if isinstance(val, np.ndarray):
        return val.tolist()
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            parsed = ast.literal_eval(val)
            if isinstance(parsed, list):
                return parsed
        except (ValueError, SyntaxError):
            pass
    return []

df["required_skills"] = df["required_skills"].apply(parse_skills)
if "nice_to_have_skills" in df.columns:
    df["nice_to_have_skills"] = df["nice_to_have_skills"].apply(parse_skills)

print(f"Total jobs: {len(df)}")
print(f"Unique companies: {df['company'].nunique()}")
print(f"Companies: {sorted(df['company'].unique())}")
print(f"Unique seniority levels: {sorted(df['seniority'].unique())}")
print(f"Unique role families: {sorted(df['role_family'].unique())}")
print(f"Null salary_min: {df['salary_min'].isna().sum()} / {len(df)}")
print(f"Null salary_max: {df['salary_max'].isna().sum()} / {len(df)}")

## 2. Basic Stats

In [ ]:
# Jobs per company — top 20
company_counts = df["company"].value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 6))
company_counts.plot.barh(ax=ax, color="#4ECDC4")
ax.set_title("Top 20 Companies by Job Count")
ax.set_xlabel("Number of Jobs")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Seniority distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

seniority_order = ["entry", "mid", "senior", "staff"]
seniority_counts = df["seniority"].value_counts().reindex(seniority_order).dropna()
seniority_counts.plot.bar(ax=axes[0], color="#45B7D1")
axes[0].set_title("Seniority Distribution")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=45)

# Location type distribution
loc_counts = df["location_type"].value_counts()
loc_counts.plot.pie(ax=axes[1], autopct="%1.1f%%", colors=["#FF6B6B", "#4ECDC4", "#45B7D1"])
axes[1].set_title("Location Type Distribution")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

## 3. Skill Analysis

In [ ]:
# Flatten all required skills
all_skills = [skill for skills in df["required_skills"] for skill in skills]
skill_counts = Counter(all_skills)

print(f"Total skill mentions: {len(all_skills)}")
print(f"Unique skills: {len(skill_counts)}")
print()

# Top 30 skills by frequency
top30 = skill_counts.most_common(30)
skill_names, skill_freqs = zip(*top30)

fig, ax = plt.subplots(figsize=(14, 8))
ax.barh(range(len(skill_names)), skill_freqs, color="#45B7D1")
ax.set_yticks(range(len(skill_names)))
ax.set_yticklabels(skill_names)
ax.set_title("Top 30 Skills by Frequency")
ax.set_xlabel("Number of Job Postings")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Skill co-occurrence heatmap (top 20 skills)
top20_skills = [s for s, _ in skill_counts.most_common(20)]
cooccurrence = pd.DataFrame(0, index=top20_skills, columns=top20_skills)

for skills in df["required_skills"]:
    present = [s for s in skills if s in top20_skills]
    for i, s1 in enumerate(present):
        for s2 in present[i + 1:]:
            cooccurrence.loc[s1, s2] += 1
            cooccurrence.loc[s2, s1] += 1

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(cooccurrence, dtype=bool))  # upper triangle only
sns.heatmap(
    cooccurrence,
    mask=mask,
    annot=True,
    fmt="d",
    cmap="YlOrRd",
    ax=ax,
    linewidths=0.5,
)
ax.set_title("Skill Co-occurrence Heatmap (Top 20 Skills)")
plt.tight_layout()
plt.show()

## 4. Salary Analysis

In [ ]:
# Salary distribution by seniority
df_salary = df.dropna(subset=["salary_min", "salary_max"]).copy()
df_salary["salary_mid"] = (df_salary["salary_min"].astype(float) + df_salary["salary_max"].astype(float)) / 2

print(f"Jobs with salary data: {len(df_salary)} / {len(df)} ({100 * len(df_salary) / len(df):.1f}%)")
print(f"Salary midpoint range: ${df_salary['salary_mid'].min():,.0f} — ${df_salary['salary_mid'].max():,.0f}")
print(f"Median salary midpoint: ${df_salary['salary_mid'].median():,.0f}")
print()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plot by seniority
seniority_order = ["entry", "mid", "senior", "staff"]
seniority_present = [s for s in seniority_order if s in df_salary["seniority"].values]
if seniority_present:
    sns.boxplot(
        data=df_salary,
        x="seniority",
        y="salary_mid",
        order=seniority_present,
        palette="coolwarm",
        ax=axes[0],
    )
    axes[0].set_title("Salary Distribution by Seniority")
    axes[0].set_ylabel("Midpoint Salary ($)")
    axes[0].tick_params(axis="x", rotation=45)
else:
    axes[0].text(0.5, 0.5, "No salary data by seniority", ha="center", va="center")

# Box plot by role_family (only include role families with salary data)
roles_with_salary = df_salary["role_family"].value_counts()
roles_with_salary = roles_with_salary[roles_with_salary >= 2].index.tolist()
if roles_with_salary:
    sns.boxplot(
        data=df_salary[df_salary["role_family"].isin(roles_with_salary)],
        x="role_family",
        y="salary_mid",
        order=sorted(roles_with_salary),
        palette="viridis",
        ax=axes[1],
    )
    axes[1].set_title("Salary Distribution by Role Family")
    axes[1].set_ylabel("Midpoint Salary ($)")
    axes[1].tick_params(axis="x", rotation=45)
else:
    axes[1].text(0.5, 0.5, "Not enough salary data by role family", ha="center", va="center")

plt.tight_layout()
plt.show()

## 5. Role Family Breakdown

In [ ]:
role_counts = df["role_family"].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette("Set2", n_colors=len(role_counts))
role_counts.plot.bar(ax=ax, color=colors)
ax.set_title("Jobs by Role Family")
ax.set_xlabel("Role Family")
ax.set_ylabel("Number of Jobs")
ax.tick_params(axis="x", rotation=45)

# Annotate bars with counts
for i, (count, label) in enumerate(zip(role_counts.values, role_counts.index)):
    ax.text(i, count + 0.5, str(count), ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics table
summary = pd.DataFrame({
    "Metric": [
        "Total jobs",
        "Unique companies",
        "Unique skills (required)",
        "Avg required skills per job",
        "Jobs with salary data",
        "Median salary (midpoint)",
        "Most common role family",
        "Most common seniority",
        "Most common location type",
    ],
    "Value": [
        len(df),
        df["company"].nunique(),
        len(skill_counts),
        f"{df['required_skills'].apply(len).mean():.1f}",
        f"{len(df_salary)} ({100 * len(df_salary) / len(df):.1f}%)",
        f"${df_salary['salary_mid'].median():,.0f}" if len(df_salary) > 0 else "N/A",
        df["role_family"].mode().iloc[0] if len(df) > 0 else "N/A",
        df["seniority"].mode().iloc[0] if len(df) > 0 else "N/A",
        df["location_type"].mode().iloc[0] if len(df) > 0 else "N/A",
    ],
})
summary.style.hide(axis="index")